# 02b - Oracle Context Evaluation

Controlled evaluation with and without source-of-truth article context. This notebook keeps the experiment separate from retrieval: oracle context is built only from the references already present in the evaluation dataset.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'pyproject.toml').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('Project root not found')
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from legal_rag.oracle_context_evaluation import OracleEvaluationConfig, build_oracle_contexts, run_oracle_context_evaluation
from legal_rag.oracle_context_evaluation.env import load_env_file, resolve_env_file
from legal_rag.oracle_context_evaluation.io import read_jsonl

ENV_FILE = resolve_env_file(ROOT / '.env') or (ROOT / '.env')
load_env_file(ENV_FILE)

EVALUATION_DIR = ROOT / 'data' / 'evaluation_clean'
LAWS_DIR = ROOT / 'data' / 'laws_dataset_clean'
OUTPUT_DIR = ROOT / 'data' / 'evaluation_runs' / 'oracle_context'

mcq_records = read_jsonl(EVALUATION_DIR / 'questions_mcq.jsonl')
no_hint_records = read_jsonl(EVALUATION_DIR / 'questions_no_hint.jsonl')
contexts = build_oracle_contexts(no_hint_records, LAWS_DIR)

print(json.dumps({
    'mcq_records': len(mcq_records),
    'no_hint_records': len(no_hint_records),
    'contexts': len(contexts),
    'context_errors': sum(1 for ctx in contexts if ctx.error),
}, ensure_ascii=False, indent=2))

{
  "mcq_records": 100,
  "no_hint_records": 100,
  "contexts": 100,
  "context_errors": 0
}


In [2]:
sample_index = 4
sample_mcq = mcq_records[sample_index]
sample_no_hint = no_hint_records[sample_index]
sample_context = contexts[sample_index]

print('MCQ question:')
print(sample_mcq['question_stem'])
print(sample_mcq['options'])
print('\nNo-hint question:')
print(sample_no_hint['question'])
print('\nOracle context article IDs:')
print(sample_context.context_article_ids)
print('\nOracle context preview:')
print(sample_context.context_text[:1200])

MCQ question:
Da quanti membri è composta la Consulta regionale per la mobilità ciclistica?
{'A': 'Tale organo non esiste', 'B': '15 membri estratti a sorte tra i cittadini', 'C': 'Dal Presidente della Giunta Regionale e da un assessore', 'D': 'Da 7 membri', 'E': 'Dai membri eletti dalla Federazione ciclistica nazionale', 'F': 'Da almeno 3 ciclisti professionisti'}

No-hint question:
Da quanti membri è composta la Consulta regionale per la mobilità ciclistica?

Oracle context article IDs:
['vda:lr:2019-10-08:16#art:4bis', 'vda:lr:2024-10-07:19#art:2']

Oracle context preview:
[1] Legge regionale 8 ottobre 2019, n. 16 - Testo vigente
Article: 4bis
(Consulta regionale per la mobilità ciclistica) (3)
1. Con decreto dell'assessore regionale competente in materia di mobilità sostenibile è istituita, presso il medesimo assessorato, la Consulta regionale per la mobilità ciclistica, quale organismo consultivo per l'attuazione di politiche e interventi relativi all'ambito della mobilità sosteni

## Run

Set `RUN_REMOTE = True` only when `UTOPIA_API_KEY` and the model endpoint are available. Smoke mode runs one selected row; full mode uses all selected rows.

In [3]:
RUN_REMOTE = bool(os.getenv('UTOPIA_API_KEY'))
SMOKE = True

if RUN_REMOTE:
    manifest = run_oracle_context_evaluation(
        OracleEvaluationConfig(
            evaluation_dir=str(EVALUATION_DIR),
            laws_dir=str(LAWS_DIR),
            output_dir=str(OUTPUT_DIR),
            env_file=str(ENV_FILE),
            api_key=os.getenv('UTOPIA_API_KEY'),
            api_url=os.getenv('UTOPIA_OLLAMA_CHAT_URL') or None,
            base_url=os.getenv('UTOPIA_BASE_URL', 'https://utopia.hpc4ai.unito.it/api'),
            chat_model=os.getenv('UTOPIA_CHAT_MODEL', 'SLURM.gpt-oss:120b'),
            judge_model=os.getenv('UTOPIA_JUDGE_MODEL') or None,
            smoke=SMOKE,
        )
    )
    print(json.dumps(manifest['counts'], ensure_ascii=False, indent=2))
else:
    print('Remote run skipped. Set RUN_REMOTE = True to execute the benchmark.')

{
  "mcq": 1,
  "no_hint": 1,
  "context_rows": 1,
  "context_errors": 0,
  "article_references": 1,
  "resolved_article_references": 1
}


In [4]:
summary_path = OUTPUT_DIR / 'oracle_context_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    rows = []
    for name in ['mcq_no_context', 'mcq_oracle_context', 'no_hint_no_context', 'no_hint_oracle_context']:
        item = summary[name]
        rows.append({
            'run': name,
            'processed': item['processed'],
            'judged': item['judged'],
            'accuracy': item['accuracy'],
            'mean_score': item['mean_score'],
            'coverage': item['coverage'],
            'strict_accuracy': item['strict_accuracy'],
            'errors': item['errors'],
        })
    display(pd.DataFrame(rows))
    display(pd.DataFrame(summary['delta_oracle_minus_no_context']['mcq']['by_level']).T)
    display(pd.DataFrame(summary['delta_oracle_minus_no_context']['no_hint']['by_level']).T)
else:
    print('No summary artifact yet.')

,run,processed,judged,accuracy,mean_score,coverage,strict_accuracy,errors
0,mcq_no_context,1,0,None,None,0.0,0.0,1
1,mcq_oracle_context,1,0,None,None,0.0,0.0,1
2,no_hint_no_context,1,0,None,None,0.0,0.0,1
3,no_hint_oracle_context,1,0,None,None,0.0,0.0,1


,accuracy,coverage,mean_score,strict_accuracy
L1,NaN,0.0,NaN,0.0


,accuracy,coverage,mean_score,strict_accuracy
L1,NaN,0.0,NaN,0.0
